# Auto‑Research AMT – Google Colab Notebook

Repo ini sudah **bersih** dan siap dijalankan di Colab.  
Notebook ini akan:
1️⃣️⃣ Install dependensi.
2️⃣️⃣ Muat data contoh (XAUUSD 1‑menit).
3️⃣️⃣ Jika Anda mengisi `LLM_API_KEY`, sinyal akan di‑generate lewat LLM (OpenAI/Anthropic).
4️⃣️⃣ Jalankan back‑test dan tampilkan metrik serta `trade_log.csv`.


In [ ]:
# 1️⃣ Clone repo (skip if already present)
%cd /content
if [ ! -d "autoresearch-trading" ]; then
  !git clone https://github.com/your-username/autoresearch-trading.git
fi
%cd autoresearch-trading


In [ ]:
# 2️⃣ Install requirements
!pip install -r requirements.txt


In [ ]:
# 3️⃣ Set LLM API key (replace with your own key)
import os
os.environ["LLM_API_KEY"] = "YOUR_API_KEY_HERE"  # <-- edit this line
os.environ["LLM_PROVIDER"] = "openai"          # or "anthropic", "groq", ...

# Optional: you can also load from a .env file if you upload one
# from dotenv import load_dotenv
# load_dotenv('.env')


In [ ]:
# 4️⃣ Load data (example XAUUSD 1‑minute parquet)
import pandas as pd
data_path = "data/XAUUSD_1m_20140114_20260626.parquet"
df = pd.read_parquet(data_path)
print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")


In [ ]:
# 5️⃣ Run backtest – set use_api=True to call the LLM
from src.backtest import run_backtest

def dummy_strategy(df, api=False):
    # The same dummy strategy from the module – kept for safety
    if api:
        # Return a prompt for the LLM (same as in src.backtest __main__)
        prompt = (
            "Berikan sinyal trading dalam format JSON untuk data OHLCV berikut. "
            "Setiap elemen harus berisi 'index' (int), 'signal' (1=LONG, -1=SHORT, 0=FLAT) dan 'reason' (string). "
            "Gunakan aturan AMT‑Adaptive‑Timeframe‑Observation: "
            "Jika close > open, sinyal LONG, jika close < open, sinyal SHORT, else FLAT. "
            f"Data: {df.head(5).to_json(orient='records')}"
        )
        return prompt
    # Local mode – simple rule‑based signals
    import numpy as np
    signals = np.where(df['close'] > df['open'], 1, -1)
    reasons = np.where(df['close'] > df['open'], "Close > Open", "Close < Open")
    return pd.DataFrame({"signal": signals, "reason": reasons}, index=df.index)

# Run the backtest – change use_api to False if you don't want LLM calls
result = run_backtest(dummy_strategy, use_api=True)
print("Metrics:")
import json
print(json.dumps(result["metrics"], indent=2))
print("Trade log saved to:", result["log_path"])

In [ ]:
# 6️⃣ Load and display the trade log (first 10 rows)
import pandas as pd
log_df = pd.read_csv(result["log_path"])
display(log_df.head(10))
